# ⚡ Train Virgo-1.0-Bayon on a free TPU — Kaggle or Colab (Keras + JAX)

Kaggle's free **TPU** (v5e-8: 8 chips, 128 GB) trains Gemma 3 12B at **full precision** (no 4-bit squeezing) and
much faster than the T4 GPUs, with its own ~20 hours a week (separate from your GPU hours). It uses Google's
**Keras + JAX** tools for Gemma (`keras_hub`), LoRA fine-tuning on the same Virgo data (your examples plus the
Gemma 27B ones, if you made them).

⚠️ **New and experimental**: this is a different training path from the GPU notebooks. If a step fails, send the
error to Claude. The GPU notebook (`chat/train_kaggle.ipynb`) stays the reliable option.

**First time only**
1. Kaggle → **Settings → Phone verification**.
2. Open Gemma 3 on Kaggle ([kaggle.com/models/google/gemma-3](https://www.kaggle.com/models/google/gemma-3)) and accept the license (**Request access**).
3. **Add-ons → Secrets**: `HF_TOKEN` (Hugging Face, **write** access), ticked for this notebook.

**Colab instead of Kaggle:** Runtime → Change runtime type → **TPU** (free: v2-8 when available; Pro: v5e/v6e).
Add Colab secrets (🔑, notebook access on): `HF_TOKEN`, plus `KAGGLE_USERNAME` and `KAGGLE_KEY` (kaggle.com → Settings →
API → Create New Token; the Keras Gemma weights download from Kaggle). On smaller TPUs (Colab's v2-8, or a single
v5e/v6e chip) the notebook trains the **4B** automatically, as **Virgo-1.0-Bayon-4B**. Then Runtime → Run all.

**Run on Kaggle**
1. **Session options**: **Accelerator → TPU VM v5e-8** (or TPU v3-8), **Internet → On**.
2. **Save Version → Save & Run All (Commit)**: it runs in the background, so you can close the tab.
3. It saves the trained model privately to Hugging Face as **Virgo-1.0-Bayon** (a whole model, ~24 GB).
   To use it, set `HF_MODEL = "<your name>/Virgo-1.0-Bayon"` in a server notebook. Compare its answers
   with the GPU-trained Virgo-1.0-Angkor 12B before switching for good.

**Virgo-1.0-Bayon** is the TPU-trained Virgo (named after the Bayon temple); the GPU-trained one stays **Virgo-1.0-Angkor**. Bayon learns to introduce itself as Virgo-1.0-Bayon.

In [ ]:
# ⚙️ Settings
PRESET = "auto"   # "auto": 12B on a big TPU (Kaggle v5e-8 / v3-8), 4B on a small one; or "gemma3_instruct_12b" / "gemma3_instruct_4b"
MODEL_NAME = "auto"   # Virgo-1.0-Bayon (12B) or Virgo-1.0-Bayon-4B, from the size
LORA_RANK = 16
EPOCHS = 2
SEQ_LEN = 1024                    # tokens per example (longer examples are cut)
BATCH = 8
LEARNING_RATE = 1e-4

In [ ]:
# 1️⃣ Set up (TPU, Keras + JAX, Virgo's data)
import os, subprocess, sys, glob, json
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "keras>=3.10", "keras-hub>=0.23", "huggingface_hub", "safetensors"], check=True)
import jax, keras, keras_hub
KAGGLE = os.path.isdir("/kaggle/working")
devices = jax.devices()
assert devices and devices[0].platform == "tpu", (f"❌ No TPU ({devices[:1]}): " + (
    "Session options → Accelerator → TPU VM v5e-8 (and Save Version with the TPU)" if KAGGLE else "Runtime → Change runtime type → TPU"))
kind = devices[0].device_kind
print("✅", len(devices), "×", kind, "| keras", keras.__version__, "| keras_hub", keras_hub.__version__)
# 12B needs about 128 GB of TPU memory in all (8 chips × 16 GB: Kaggle's v5e-8 / v3-8); smaller TPUs get the 4B.
big_tpu = len(devices) >= 8 and not any(v in kind.lower() for v in ("v2",))
if PRESET == "auto":
    PRESET = "gemma3_instruct_12b" if big_tpu else "gemma3_instruct_4b"
if MODEL_NAME == "auto":
    MODEL_NAME = "Virgo-1.0-Bayon" if "12b" in PRESET else "Virgo-1.0-Bayon-4B"
print("Training", PRESET, "→", MODEL_NAME)

def secret(name):
    """Kaggle secrets, Colab secrets, the environment, or asked once."""
    if KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    from getpass import getpass
    return getpass(f"{name}: ").strip() or None

if not KAGGLE:  # outside Kaggle, the Keras Gemma weights download with your Kaggle API key
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        value = secret(name)
        assert value, f"❌ Add the {name} secret (kaggle.com → Settings → API → Create New Token)"
        os.environ[name] = value

WORK = "/kaggle/working" if KAGGLE else "/content"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"], check=True)
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
from huggingface_hub import login, HfApi, hf_hub_download
login(token=secret("HF_TOKEN"))
ME = HfApi().whoami()["name"]
REPO = f"{ME}/{MODEL_NAME}"
try:  # extra examples made by chat/distill_vertex.ipynb (Gemma 3 27B)
    hf_hub_download(f"{ME}/Virgo-1.0-Angkor-Data", "distilled_gemma27b.jsonl", repo_type="dataset", local_dir="chat/data")
except Exception:
    pass

# Each example → the conversation so far (prompt) and Virgo's last reply (response), in Gemma's chat format.
# Gemma has no system role: Virgo's instructions go at the start of the first user turn.
def turn(role, text):
    return f"<start_of_turn>{'model' if role == 'assistant' else 'user'}\n{text}<end_of_turn>\n"
prompts, responses = [], []
for path in sorted(glob.glob("chat/data/*.jsonl")):
    for line in open(path, encoding="utf-8"):
        if not line.strip():
            continue
        msgs = json.loads(line)["messages"]
        system = " ".join(m["content"] for m in msgs if m["role"] == "system")
        talk = [m for m in msgs if m["role"] != "system"]
        if len(talk) < 2 or talk[-1]["role"] != "assistant":
            continue
        history = ""
        for i, m in enumerate(talk[:-1]):
            text = f"{system}\n\n{m['content']}" if i == 0 and system else m["content"]
            history += turn(m["role"], text)
        name = MODEL_NAME.replace("-4B", "")  # the data says Virgo-1.0-Angkor: this model is Virgo-1.0-Bayon
        prompts.append((history + "<start_of_turn>model\n").replace("Virgo-1.0-Angkor", name))
        responses.append((talk[-1]["content"] + "<end_of_turn>").replace("Virgo-1.0-Angkor", name))
print("✅", len(prompts), "training examples")

In [ ]:
# 2️⃣ Load Gemma 3 across the TPU chips, with LoRA
mesh = keras.distribution.DeviceMesh((1, len(devices)), ["batch", "model"], devices=keras.distribution.list_devices())
# How each weight is split across the chips (the "model" axis). Newer keras_hub has a ready-made map for
# Gemma 3; otherwise this one, for Gemma's layer names. Weights it doesn't match are copied to every chip.
def gemma_layout_map(mesh):
    for cls in ("Gemma3Backbone", "GemmaBackbone"):
        get = getattr(getattr(keras_hub.models, cls, None), "get_layout_map", None)
        if cls == "Gemma3Backbone" and get:
            return get(mesh)
    m, b = "model", "batch"
    layout = keras.distribution.LayoutMap(mesh)
    layout["token_embedding/embeddings"] = (m, b)
    layout["decoder_block.*attention.*(query|key|value).kernel"] = (m, b, None)
    layout["decoder_block.*attention_output.kernel"] = (m, None, b)
    layout["decoder_block.*ffw_gating.*kernel"] = (b, m)
    layout["decoder_block.*ffw_linear.kernel"] = (m, b)
    return layout
layout_map = gemma_layout_map(mesh)
keras.distribution.set_distribution(keras.distribution.ModelParallel(layout_map=layout_map, batch_dim_name="batch"))

gemma = keras_hub.models.Gemma3CausalLM.from_preset(PRESET, dtype="bfloat16")
gemma.backbone.enable_lora(rank=LORA_RANK)
gemma.preprocessor.sequence_length = SEQ_LEN
optimizer = keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=0.01)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), optimizer=optimizer,
              weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()])
gemma.summary()

In [ ]:
# 3️⃣ Train (the loss should go down; on a v5e-8 this takes well under an hour for ~2,000–5,000 examples)
history = gemma.fit({"prompts": prompts, "responses": responses}, epochs=EPOCHS, batch_size=BATCH)

In [ ]:
# 4️⃣ Try Virgo
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់", "Translate to Khmer: Thank you for your help."]:
    out = gemma.generate(f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n", max_length=384, strip_prompt=True)
    print("🧑", q); print("🤖", out.split("<end_of_turn>")[0].strip(), "\n")

In [ ]:
# 5️⃣ Save to Hugging Face (private): as a normal Transformers model the Virgo server can load
import shutil
out = f"{WORK}/{MODEL_NAME}"
shutil.rmtree(out, ignore_errors=True)
saved = None
try:
    gemma.export_to_transformers(out)  # merges LoRA into the weights; needs a recent keras_hub
    saved = "transformers"
except Exception as err:
    print("⚠️ Transformers export not available here:", repr(err)[:300])
if saved is None:
    # Keep the work anyway, in Keras format (Claude can convert it later).
    gemma.save_to_preset(out)
    saved = "keras"
open(f"{out}/README.md", "w").write(f"""---
license: gemma
language: [en, km]
tags: [virgo, ksn, khmer]
---
# {MODEL_NAME}

KSN's {MODEL_NAME}, fine-tuned (LoRA, merged) from Google's Gemma 3 ({PRESET}) on a TPU with Keras.
Use is subject to the [Gemma Terms of Use](https://ai.google.dev/gemma/terms). Format: {saved}.
""")
HfApi().create_repo(REPO, private=True, exist_ok=True)
HfApi().upload_folder(folder_path=out, repo_id=REPO, commit_message=f"{MODEL_NAME} ({saved})")
print(f"✅ Saved ({saved} format): https://huggingface.co/{REPO}")
if saved == "transformers":
    print(f'To use it: set HF_MODEL = "{REPO}" in a server notebook.')
else:
    print("Send this message to Claude: the model is saved in Keras format and needs converting before the server can use it.")